In [2]:
!pip uninstall -y flash-attn flash_attn
!pip install -q transformers==4.41.2 accelerate sentencepiece

Found existing installation: flash_attn 2.4.2
Uninstalling flash_attn-2.4.2:
  Successfully uninstalled flash_attn-2.4.2

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
import time
import psutil
import torch
from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TEST_TEXT = """
Machine learning techniques are increasingly used for phishing detection.
Transformers provide strong contextual understanding.
"""

print("Using Device:", DEVICE)

# =========================
# Load BERT
# =========================

print("\nLoading BERT...")

bert_name = "bert-base-uncased"

bert_tokenizer = AutoTokenizer.from_pretrained(bert_name)

bert_model = AutoModel.from_pretrained(
    bert_name
).to(DEVICE)

# =========================
# Load GPT-2
# =========================

print("Loading GPT-2...")

gpt_name = "gpt2"

gpt_tokenizer = AutoTokenizer.from_pretrained(gpt_name)

if gpt_tokenizer.pad_token is None:
    gpt_tokenizer.pad_token = gpt_tokenizer.eos_token

gpt_model = AutoModelForCausalLM.from_pretrained(
    gpt_name
).to(DEVICE)

# =========================
# Benchmark Function
# =========================

def benchmark(model, tokenizer, text, model_type):

    process = psutil.Process()

    memory_before = process.memory_info().rss / 1024**2

    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

    start = time.time()

    with torch.no_grad():

        if model_type == "BERT":
            _ = model(**inputs)

        else:
            _ = model.generate(
                inputs["input_ids"],
                max_new_tokens=50
            )

    end = time.time()

    memory_after = process.memory_info().rss / 1024**2

    tokens = inputs["input_ids"].shape[1]

    return {
        "Latency(sec)": round(end - start, 4),
        "Tokens": tokens,
        "Memory(MB)": round(memory_after - memory_before, 2),
        "Tokens/sec": round(tokens / (end - start), 2)
    }

# =========================
# Run Benchmarks
# =========================

bert_results = benchmark(
    bert_model,
    bert_tokenizer,
    TEST_TEXT,
    "BERT"
)

gpt_results = benchmark(
    gpt_model,
    gpt_tokenizer,
    TEST_TEXT,
    "GPT"
)

# =========================
# Display Results
# =========================

print("\n==============================")
print("BERT RESULTS")
print("==============================")

for k, v in bert_results.items():
    print(f"{k}: {v}")

print("\n==============================")
print("GPT-2 RESULTS")
print("==============================")

for k, v in gpt_results.items():
    print(f"{k}: {v}")

print("\n==============================")
print("MODEL INFORMATION")
print("==============================")

print("BERT Context Window : 512")
print("GPT-2 Context Window: 1024")
print("Device:", DEVICE)

Using Device: cuda

Loading BERT...
Loading GPT-2...


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



BERT RESULTS
Latency(sec): 0.27
Tokens: 20
Memory(MB): 42.1
Tokens/sec: 74.07

GPT-2 RESULTS
Latency(sec): 0.2862
Tokens: 21
Memory(MB): 105.0
Tokens/sec: 73.38

MODEL INFORMATION
BERT Context Window : 512
GPT-2 Context Window: 1024
Device: cuda
